In [1]:
!pip install -q sentence-transformers faiss-cpu nltk rank_bm25

import requests, json, time, numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader
import faiss, nltk, string

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.1 MB/s eta 0:00:00:00:0100:01


/tmp/ipykernel_58/2920897860.py:6: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, losses


In [4]:
import numpy as np
import json

with open('/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/extraction_log.json') as f:
    extraction_log = json.load(f)

# Check what scores are in the results
rrf_scores = []
bge_scores = []

for item in extraction_log[:100]:
    if 'retrieved' in item:
        for r in item['retrieved']:
            if 'rrf_score' in r:
                rrf_scores.append(r['rrf_score'])
            if 'bge_score' in r:
                bge_scores.append(r['bge_score'])

if rrf_scores:
    print(f"RRF scores range: {min(rrf_scores):.6f} to {max(rrf_scores):.6f}")
    print(f"RRF mean: {np.mean(rrf_scores):.6f}")
if bge_scores:
    print(f"BGE scores range: {min(bge_scores):.6f} to {max(bge_scores):.6f}")
    print(f"BGE mean: {np.mean(bge_scores):.6f}")

RRF scores range: 0.014493 to 0.032787
RRF mean: 0.026015
BGE scores range: 0.000000 to 0.000000
BGE mean: 0.000000


In [6]:
import numpy as np
from tqdm import tqdm

def evaluate(test_data, retrieve_fn, top_n=100):
    recall_1, recall_5, recall_10, map_10 = [], [], [], []
    for item in tqdm(test_data):
        query = item['Question']
        gt_pids = set(
            (p['DocumentID'], p['PassageID'])
            for p in item['Passages']
        )
        results = retrieve_fn(query, top_n=top_n)
        retrieved = [(r['DocumentID'], r['PassageID']) for r in results]
        recall_1.append(1.0 if any(p in gt_pids for p in retrieved[:1]) else 0.0)
        recall_5.append(1.0 if any(p in gt_pids for p in retrieved[:5]) else 0.0)
        recall_10.append(1.0 if any(p in gt_pids for p in retrieved[:10]) else 0.0)
        ap, hits = 0.0, 0
        for k, pid in enumerate(retrieved[:10], 1):
            if pid in gt_pids:
                hits += 1
                ap += hits / k
        map_10.append(ap / max(len(gt_pids), 1))
    return {
        'Recall@1':  np.mean(recall_1),
        'Recall@5':  np.mean(recall_5),
        'Recall@10': np.mean(recall_10),
        'MAP@10':    np.mean(map_10)
    }

# Also need inner_decay and other helpers
def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

# Now run
print("Running BGE+CE no window on n=200...")
r_nowindow = evaluate(test_data[:200], retrieve_bge_ce_nowindow)
print(f"BGE+CE no window: {r_nowindow}")

print("\nRunning BGE+CE+Window on n=200...")
r_window = evaluate(test_data[:200], retrieve_full_v2)
print(f"BGE+CE+Window: {r_window}")

print("\n=== Comparison ===")
print(f"{'Metric':<12} {'No Window':>12} {'Window':>10}")
print("-" * 36)
for m in ['Recall@1', 'Recall@5', 'Recall@10', 'MAP@10']:
    print(f"{m:<12} {r_nowindow[m]:>12.4f} {r_window[m]:>10.4f}")

Running BGE+CE no window on n=200...


NameError: name 'test_data' is not defined

In [6]:
import requests, json, time, numpy as np, string, pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import faiss, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Load ObliQA data ──────────────────────────────────────────
print("Loading ObliQA...")
test_data = json.loads(requests.get(
    "https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_test.json"
).text)
print(f"Test questions: {len(test_data)}")

all_passages = []
for doc_id in range(1, 41):
    r = requests.get(
        f"https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/StructuredRegulatoryDocuments/{doc_id}.json"
    )
    if r.status_code == 200:
        all_passages.extend(json.loads(r.text))
    time.sleep(0.2)
print(f"Raw passages: {len(all_passages)}")

# ── Clean ObliQA passages ─────────────────────────────────────
clean_passages = []
for p in all_passages:
    text = p['Passage'].strip()
    if len(text) < 30:
        continue
    if len(text) > 1500:
        start = 0
        while start < len(text):
            chunk = text[start:start+1000]
            if len(chunk) < 100 and start > 0:
                break
            clean_passages.append({
                'DocumentID': p['DocumentID'],
                'PassageID':  p['PassageID'],
                'Passage':    chunk,
            })
            start += 800
    else:
        clean_passages.append({
            'DocumentID': p['DocumentID'],
            'PassageID':  p['PassageID'],
            'Passage':    text,
        })
print(f"Clean passages: {len(clean_passages)}")

# ── Build BGE index ───────────────────────────────────────────
print("Building BGE index...")
bge_model = SentenceTransformer('BAAI/bge-base-en-v1.5')
embeddings = bge_model.encode(
    [p['Passage'] for p in clean_passages],
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype(np.float32))
print(f"FAISS index: {index.ntotal}")

# ── Build BM25 ────────────────────────────────────────────────
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

corpus = [tokenize(p['Passage']) for p in clean_passages]
bm25 = BM25Okapi(corpus)
print("BM25 ready.")

# ── Load CrossEncoder ─────────────────────────────────────────
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("All ready.")

# ── Build PassageID lookup ────────────────────────────────────
pid_to_idx = {}
for i, p in enumerate(clean_passages):
    pid = (p['DocumentID'], p['PassageID'])
    if pid not in pid_to_idx:
        pid_to_idx[pid] = i

# ── Helper functions ──────────────────────────────────────────
def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window(doc_id, passage_idx):
    doc_passages = [(i, p) for i, p in enumerate(clean_passages)
                    if p['DocumentID'] == doc_id]
    current_pos = next(
        (pos for pos, (i, p) in enumerate(doc_passages)
         if i == passage_idx), None
    )
    if current_pos is None:
        return clean_passages[passage_idx]['Passage']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_passages):
            window.append(doc_passages[pos][1]['Passage'])
    return ' '.join(window)

def get_rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    _, bge_indices = index.search(qe, top_n)
    bge_indices = bge_indices[0]
    bm25_raw = bm25.get_scores(tokenize(query))
    bm25_indices = np.argsort(bm25_raw)[::-1][:top_n]
    rrf_scores = {}
    for rank, idx in enumerate(bge_indices):
        pid = (clean_passages[idx]['DocumentID'],
               clean_passages[idx]['PassageID'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        pid = (clean_passages[idx]['DocumentID'],
               clean_passages[idx]['PassageID'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(
        rrf_scores.items(), key=lambda x: x[1], reverse=True
    )
    return sorted_pids, rrf_scores

def evaluate(test_data, retrieve_fn, top_n=100):
    recall_1, recall_5, recall_10, map_10 = [], [], [], []
    for item in tqdm(test_data):
        query = item['Question']
        gt_pids = set(
            (p['DocumentID'], p['PassageID'])
            for p in item['Passages']
        )
        results = retrieve_fn(query, top_n=top_n)
        retrieved = [(r['DocumentID'], r['PassageID']) for r in results]
        recall_1.append(1.0 if any(p in gt_pids for p in retrieved[:1]) else 0.0)
        recall_5.append(1.0 if any(p in gt_pids for p in retrieved[:5]) else 0.0)
        recall_10.append(1.0 if any(p in gt_pids for p in retrieved[:10]) else 0.0)
        ap, hits = 0.0, 0
        for k, pid in enumerate(retrieved[:10], 1):
            if pid in gt_pids:
                hits += 1
                ap += hits / k
        map_10.append(ap / max(len(gt_pids), 1))
    return {
        'Recall@1':  np.mean(recall_1),
        'Recall@5':  np.mean(recall_5),
        'Recall@10': np.mean(recall_10),
        'MAP@10':    np.mean(map_10)
    }

# ── Retrieval functions ───────────────────────────────────────
def retrieve_bge_ce_nowindow(query, top_n=100):
    """BGE + CE on current passage only. No window."""
    qe = bge_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(qe, top_n)
    scores, indices = scores[0], indices[0]
    results, seen = [], set()
    for rank_idx in range(min(top_n, len(indices))):
        idx = indices[rank_idx]
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid in seen:
            continue
        seen.add(pid)
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce = float(max(curr_ce))
        kept = inner_decay(curr_sents, curr_ce)
        if kept:
            results.append({
                'DocumentID':   p['DocumentID'],
                'PassageID':    p['PassageID'],
                'top_ce_score': top_ce,
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_bge_ce_window(query, top_n=100):
    """BGE + CE + Window."""
    qe = bge_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(qe, top_n)
    scores, indices = scores[0], indices[0]
    results, seen = [], set()
    for rank_idx in range(min(top_n, len(indices))):
        idx = indices[rank_idx]
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid in seen:
            continue
        seen.add(pid)
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce = float(max(curr_ce))
        window = get_window(p['DocumentID'], idx)
        win_sents = sent_tokenize(window)
        win_ce = cross_encoder.predict([[query, s] for s in win_sents])
        kept = inner_decay(win_sents, win_ce)
        if kept:
            results.append({
                'DocumentID':   p['DocumentID'],
                'PassageID':    p['PassageID'],
                'top_ce_score': top_ce,
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def make_fixed_k_retriever(k):
    """Full pipeline with fixed top-k candidates."""
    def retrieve_fixed_k(query, top_n=100):
        sorted_pids, rrf_scores = get_rrf_candidates(query, top_n)
        candidates = sorted_pids[:k]
        results, seen = [], set()
        for pid, rrf_score in candidates:
            if pid in seen:
                continue
            seen.add(pid)
            idx = pid_to_idx.get(pid)
            if idx is None:
                continue
            p = clean_passages[idx]
            curr_sents = sent_tokenize(p['Passage'])
            if not curr_sents:
                continue
            curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
            top_ce = float(max(curr_ce))
            window = get_window(p['DocumentID'], idx)
            win_sents = sent_tokenize(window)
            win_ce = cross_encoder.predict([[query, s] for s in win_sents])
            kept = inner_decay(win_sents, win_ce)
            if kept:
                results.append({
                    'DocumentID':   p['DocumentID'],
                    'PassageID':    p['PassageID'],
                    'top_ce_score': top_ce,
                })
        results.sort(key=lambda x: x['top_ce_score'], reverse=True)
        return results
    return retrieve_fixed_k

# ── Run all ablations on n=200 ────────────────────────────────
N = 200
results = {}
print(f"\nRunning ablations on n={N}...")

print("\n1. BGE + CE (no window)...")
results['bge_ce_nowindow'] = evaluate(test_data[:N], retrieve_bge_ce_nowindow)
print(results['bge_ce_nowindow'])

print("\n2. BGE + CE + Window...")
results['bge_ce_window'] = evaluate(test_data[:N], retrieve_bge_ce_window)
print(results['bge_ce_window'])

print("\n3. Fixed top-k=1...")
results['fixed_k1'] = evaluate(test_data[:N], make_fixed_k_retriever(1))
print(results['fixed_k1'])

print("\n4. Fixed top-k=5...")
results['fixed_k5'] = evaluate(test_data[:N], make_fixed_k_retriever(5))
print(results['fixed_k5'])

print("\n5. Fixed top-k=10...")
results['fixed_k10'] = evaluate(test_data[:N], make_fixed_k_retriever(10))
print(results['fixed_k10'])

# ── Full ablation table ───────────────────────────────────────
print(f"\n{'Configuration':<30} {'R@1':>6} {'R@5':>6} {'R@10':>6} {'MAP':>6}")
print("=" * 55)
known = {
    'BGE only':        (0.44, 0.65, 0.75, 0.514),
    'BM25 only':       (0.52, 0.79, 0.82, 0.578),
    'Full pipeline':   (0.55, 0.73, 0.74, 0.580),
    'Rank by RRF':     (0.32, 0.70, 0.79, 0.449),
    'Rank by win max': (0.49, 0.71, 0.73, 0.533),
}
for name, (r1, r5, r10, m) in known.items():
    print(f"{name:<30} {r1:>6.2f} {r5:>6.2f} {r10:>6.2f} {m:>6.3f}")
print("-" * 55)
labels = {
    'bge_ce_nowindow': 'BGE + CE (no window)',
    'bge_ce_window':   'BGE + CE + Window',
    'fixed_k1':        'Full + fixed top-1',
    'fixed_k5':        'Full + fixed top-5',
    'fixed_k10':       'Full + fixed top-10',
}
for key, label in labels.items():
    r = results[key]
    print(f"{label:<30} {r['Recall@1']:>6.2f} "
          f"{r['Recall@5']:>6.2f} "
          f"{r['Recall@10']:>6.2f} "
          f"{r['MAP@10']:>6.3f}")

# ── Save ──────────────────────────────────────────────────────
with open('/kaggle/working/ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: ablation_results.json")

Loading ObliQA...
Test questions: 2786
Raw passages: 13732
Clean passages: 13244
Building BGE index...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

FAISS index: 13244
BM25 ready.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

... (8 outputs left out)

100%|██████████| 200/200 [00:48<00:00,  4.16it/s]


{'Recall@1': np.float64(0.49), 'Recall@5': np.float64(0.635), 'Recall@10': np.float64(0.635), 'MAP@10': np.float64(0.507611111111111)}

5. Fixed top-k=10...


100%|██████████| 200/200 [01:06<00:00,  2.99it/s]

{'Recall@1': np.float64(0.53), 'Recall@5': np.float64(0.74), 'Recall@10': np.float64(0.8), 'MAP@10': np.float64(0.5703571428571429)}

Configuration                     R@1    R@5   R@10    MAP
BGE only                         0.44   0.65   0.75  0.514
BM25 only                        0.52   0.79   0.82  0.578
Full pipeline                    0.55   0.73   0.74  0.580
Rank by RRF                      0.32   0.70   0.79  0.449
Rank by win max                  0.49   0.71   0.73  0.533
-------------------------------------------------------
BGE + CE (no window)             0.50   0.72   0.77  0.555
BGE + CE + Window                0.50   0.72   0.77  0.555
Full + fixed top-1               0.06   0.06   0.06  0.052
Full + fixed top-5               0.49   0.64   0.64  0.508
Full + fixed top-10              0.53   0.74   0.80  0.570

Saved: ablation_results.json


In [3]:
print(clean_passages[0].keys())
print(clean_passages[0])

dict_keys(['doc_name', 'filename', 'chunk_idx', 'text'])
{'doc_name': 'Cyber Security in Power Sector 2021', 'filename': '24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf', 'chunk_idx': 0, 'text': 'CEA (Cyber Security in Power Sector) Guidelines, 2021 1.0 Background 1.1 Cyber intrusion attempts and Cyber-attacks in any critical sector are carried out with a malicious intent. In Power Sector it’s either to compromise the Power Supply System or to render the grid operation in-secure. Any such compromise, may result in maloperations of equipments, equipment damages or even in a cascading grid brownout/blackout. The much hyped air gap myth between IT and OT Systems now stands shattered. The artificial air gap created by deploying firewalls between any IT and OT System can be jumped by any insider or an outsider through social engineering. Cyber-attacks are staged through tactics & techniques of Initial Access, Execution, Persistence, Privilege Escalation, Defence Evasion, Com

CO2 calculation

In [4]:
# ---------------------------------------------------------------------
# Fills in the FIVE configs that were hardcoded in `known = {...}`:
#   BGE only, BM25 only, Full pipeline, Rank by RRF, Rank by window max
# Assumes bge_model, bge_index, bm25, cross_encoder, clean_passages,
# get_window(), sent_tokenize, inner_decay(), evaluate(), and
# test_data already exist in your notebook session (from earlier cells).
# VERIFY every function/variable name below matches your actual code —
# I'm reconstructing bge_search/bm25_search/rrf_fusion from fragments,
# not from your real implementation.
# ---------------------------------------------------------------------

import numpy as np, string, json
from tqdm import tqdm

RRF_K = 60  # matches Eq.1 / Table 2 in the paper — confirm this matches your actual constant

# Re-added because it's only defined in an earlier cell — no need to rerun
# index/embedding building, bge_index/bm25/cross_encoder/clean_passages
# should still be alive in this kernel session.
def evaluate(test_data, retrieve_fn, top_n=100):
    recall_1, recall_5, recall_10, map_10 = [], [], [], []
    for item in tqdm(test_data):
        query = item['Question']
        gt_pids = set(
            (p['DocumentID'], p['PassageID'])
            for p in item['Passages']
        )
        results = retrieve_fn(query, top_n=top_n)
        retrieved = [(r['DocumentID'], r['PassageID']) for r in results]
        recall_1.append(1.0 if any(p in gt_pids for p in retrieved[:1]) else 0.0)
        recall_5.append(1.0 if any(p in gt_pids for p in retrieved[:5]) else 0.0)
        recall_10.append(1.0 if any(p in gt_pids for p in retrieved[:10]) else 0.0)
        ap, hits = 0.0, 0
        for k, pid in enumerate(retrieved[:10], 1):
            if pid in gt_pids:
                hits += 1
                ap += hits / k
        map_10.append(ap / max(len(gt_pids), 1))
    return {
        'Recall@1':  np.mean(recall_1),
        'Recall@5':  np.mean(recall_5),
        'Recall@10': np.mean(recall_10),
        'MAP@10':    np.mean(map_10)
    }

def bge_search(query, k=100):
    q_emb = bge_model.encode([query], normalize_embeddings=True)
    _, idxs = bge_index.search(np.array(q_emb, dtype='float32'), k)
    return idxs[0].tolist()

def bm25_search(query, k=100):
    tokens = query.lower().translate(str.maketrans('', '', string.punctuation)).split()
    scores = bm25.get_scores(tokens)
    return np.argsort(scores)[::-1][:k].tolist()

def rrf_fusion(dense_idxs, sparse_idxs, k_const=RRF_K):
    scores = {}
    for rank, idx in enumerate(dense_idxs, 1):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k_const + rank)
    for rank, idx in enumerate(sparse_idxs, 1):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k_const + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def retrieve_bge_only(query, top_n=100):
    idxs = bge_search(query, k=top_n)
    return [{'DocumentID': clean_passages[i]['DocumentID'],
              'PassageID': clean_passages[i]['PassageID']} for i in idxs]

def retrieve_bm25_only(query, top_n=100):
    idxs = bm25_search(query, k=top_n)
    return [{'DocumentID': clean_passages[i]['DocumentID'],
              'PassageID': clean_passages[i]['PassageID']} for i in idxs]

def retrieve_full_pipeline(query, top_n=100):
    dense_idxs = bge_search(query, k=top_n)
    sparse_idxs = bm25_search(query, k=top_n)
    fused = rrf_fusion(dense_idxs, sparse_idxs)[:top_n]
    out = []
    for idx, _ in fused:
        p = clean_passages[idx]
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        own_score = float(max(curr_ce))  # rank-score = own-passage max, Eq.5
        out.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'], 'score': own_score})
    out.sort(key=lambda x: -x['score'])
    return out

def retrieve_rank_by_rrf(query, top_n=100):
    # Ablation: skip CE reranking entirely, rank by fused RRF score only
    dense_idxs = bge_search(query, k=top_n)
    sparse_idxs = bm25_search(query, k=top_n)
    fused = rrf_fusion(dense_idxs, sparse_idxs)[:top_n]
    return [{'DocumentID': clean_passages[i]['DocumentID'],
              'PassageID': clean_passages[i]['PassageID']} for i, _ in fused]

def retrieve_rank_by_window_max(query, top_n=100):
    # Ablation: rank by max CE score over the WHOLE window, not own-passage only
    dense_idxs = bge_search(query, k=top_n)
    sparse_idxs = bm25_search(query, k=top_n)
    fused = rrf_fusion(dense_idxs, sparse_idxs)[:top_n]
    out = []
    for idx, _ in fused:
        p = clean_passages[idx]
        window = get_window(p['DocumentID'], idx)
        win_sents = sent_tokenize(window)
        if not win_sents:
            continue
        win_ce = cross_encoder.predict([[query, s] for s in win_sents])
        out.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                     'score': float(max(win_ce))})
    out.sort(key=lambda x: -x['score'])
    return out


# ── Run on the SAME n=200 sample used for the other five configs ──────
N = 200
missing_results = {}

print("6. BGE only...")
missing_results['bge_only'] = evaluate(test_data[:N], retrieve_bge_only)
print(missing_results['bge_only'])

print("\n7. BM25 only...")
missing_results['bm25_only'] = evaluate(test_data[:N], retrieve_bm25_only)
print(missing_results['bm25_only'])

print("\n8. Full pipeline...")
missing_results['full_pipeline'] = evaluate(test_data[:N], retrieve_full_pipeline)
print(missing_results['full_pipeline'])

print("\n9. Rank by RRF (no CE)...")
missing_results['rank_by_rrf'] = evaluate(test_data[:N], retrieve_rank_by_rrf)
print(missing_results['rank_by_rrf'])

print("\n10. Rank by window max...")
missing_results['rank_by_window_max'] = evaluate(test_data[:N], retrieve_rank_by_window_max)
print(missing_results['rank_by_window_max'])

# ── Merge with the previously-verified five and save ───────────────────
with open('/kaggle/working/ablation_results.json') as f:
    verified_results = json.load(f)

all_results = {**verified_results, **missing_results}
with open('/kaggle/working/ablation_results_complete.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=float)

print("\nSaved: ablation_results_complete.json (10/10 configs, all computed)")


6. BGE only...


NameError: name 'test_data' is not defined

In [5]:
!pip install -q pymupdf python-docx

import pymupdf
import docx as python_docx
import re
import os
import shutil

os.makedirs('/kaggle/working/regulationqa_cleaned_texts', exist_ok=True)

doc_path = '/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis/'

doc_name_map = {
    '24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf': 
        'Cyber_Security_Power_Sector_2021',
    'CELEX_32016R0679_EN_TXT.pdf':               'GDPR_2016',
    'CERT-In_Directions.docx':                   'CERT-In_Directions_2022',
    'Cyber Security Framework in Banks.docx':    'RBI_Cyber_Security_Framework',
    'DPDP ACT.pdf':                              'DPDP_Act_2023',
    'DPDP_Rules_2025_English_only.pdf':          'DPDP_Rules_2025',
    'Electricity Act 2003.pdf':                  'Electricity_Act_2003',
    'ISO_IEC-270012022-ed.3.pdf':                'ISO_IEC_27001_2022',
    'IT ACT 2000.pdf':                           'IT_Act_2000',
    'NCIIPC-Rules-notification-1.pdf':           'IT_Protected_System_Rules_2018',
    'NCIIPC_Guidelines_V2.pdf':                  'NCIIPC_Guidelines',
    'NIST.SP.800-66r2.pdf':                      'NIST_SP_800-66r2',
    'OECD-LEGAL-0479-en.docx':                   'OECD_Digital_Security_Guidelines',
    'Presentation - CSCRF.pdf':                  'SEBI_CSCRF',
    'State Data Centre security guidelines.pdf': 'State_DC_Security_Guidelines',
    'e-Governance Policy Initiatives.pdf':       'e-Governance_Policy_India',
    'guidelinesgovtentities.pdf':                'NCIIPC_Guidelines_Govt_Entities',
    'nist cyber security framework.docx':        'NIST_Cybersecurity_Framework',
    'pcissc_overview.pdf':                       'PCI_Security_Standards',
    'trai_takshalia.pdf':                        'TRAI_Telecom_Advisory',
}

def clean_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def extract_pdf(filepath):
    doc = pymupdf.open(filepath)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def extract_docx(filepath):
    doc = python_docx.Document(filepath)
    return "\n".join([para.text for para in doc.paragraphs])

print("Extracting cleaned text files...")
print(f"{'Document':<45} {'Words':>8}")
print("-" * 55)

success = 0
for fname, clean_name in doc_name_map.items():
    fpath = os.path.join(doc_path, fname)
    if not os.path.exists(fpath):
        print(f"{clean_name:<45} NOT FOUND")
        continue
    try:
        if fname.endswith('.pdf'):
            text = extract_pdf(fpath)
        else:
            text = extract_docx(fpath)
        text = clean_text(text)
        words = len(text.split())
        out_path = f'/kaggle/working/regulationqa_cleaned_texts/{clean_name}.txt'
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"{clean_name:<45} {words:>8,} ✓")
        success += 1
    except Exception as e:
        print(f"{clean_name:<45} ERROR: {e}")

print(f"\nDone: {success}/20 documents")

# Zip
shutil.make_archive(
    '/kaggle/working/regulationqa_cleaned_texts',
    'zip',
    '/kaggle/working',
    'regulationqa_cleaned_texts'
)
size = os.path.getsize('/kaggle/working/regulationqa_cleaned_texts.zip')
print(f"Zip: {size/1024/1024:.1f} MB")
print("Download from Output tab.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.5 MB/s eta 0:00:00
Extracting cleaned text files...
Document                                         Words
-------------------------------------------------------
Cyber_Security_Power_Sector_2021                 8,240 ✓
GDPR_2016                                       55,799 ✓
CERT-In_Directions_2022                          1,844 ✓
RBI_Cyber_Security_Framework                     6,732 ✓
DPDP_Act_2023                                   10,685 ✓
DPDP_Rules_2025                                 10,293 ✓
Electricity_Act_2003                            41,155 ✓
ISO_IEC_27001_2022                               7,194 ✓
IT_Act_2000                                     19,525 ✓
IT_Protected_System_Rules_2018                   4,044 ✓
NCIIPC_Guidelines                               14,696 ✓
NIST_SP_800-66r2                                42,128 ✓
O

In [1]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk

import pickle, faiss, json, numpy as np, string
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import nltk, pandas as pd
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Load artifacts ────────────────────────────────────────────
print("Loading artifacts...")
with open('/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/chunks.pkl', 'rb') as f:
    all_chunks = pickle.load(f)

index = faiss.read_index(
    '/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/faiss_index.bin'
)
with open('/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/bm25.pkl', 'rb') as f:
    bm25 = pickle.load(f)

rqa_df = pd.read_csv(
    '/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv'
)
print(f"Chunks: {len(all_chunks)}, Questions: {len(rqa_df)}")

# ── Models ────────────────────────────────────────────────────
bge_model    = SentenceTransformer('BAAI/bge-base-en-v1.5')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Models ready.")

# ── Helpers ───────────────────────────────────────────────────
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

def token_f1(prediction, ground_truth):
    pred = prediction.lower().split()
    gt   = ground_truth.lower().split()
    common = set(pred) & set(gt)
    if not common:
        return 0.0
    p = len(common) / len(pred)
    r = len(common) / len(gt)
    return 2 * p * r / (p + r)

def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window(doc_name, passage_idx):
    doc_chunks = [(i, c) for i, c in enumerate(all_chunks)
                  if c['doc_name'] == doc_name]
    current_pos = next(
        (pos for pos, (i, c) in enumerate(doc_chunks)
         if i == passage_idx), None
    )
    if current_pos is None:
        return all_chunks[passage_idx]['text']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_chunks):
            window.append(doc_chunks[pos][1]['text'])
    return ' '.join(window)

def get_rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    _, bge_indices = index.search(qe, top_n)
    bge_indices = bge_indices[0]
    bm25_scores  = bm25.get_scores(tokenize(query))
    bm25_indices = np.argsort(bm25_scores)[::-1][:top_n]
    rrf_scores = {}
    for rank, idx in enumerate(bge_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(
        rrf_scores.items(), key=lambda x: x[1], reverse=True
    )[:20]
    pid_to_idx = {}
    for i, c in enumerate(all_chunks):
        pid = (c['doc_name'], c['chunk_idx'])
        if pid not in pid_to_idx:
            pid_to_idx[pid] = i
    return sorted_pids, pid_to_idx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.2 MB/s eta 0:00:00:00:0100:01
Loading artifacts...
Chunks: 766, Questions: 1002


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (13 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Models ready.


In [1]:


# ── WITH window (existing full pipeline) ─────────────────────
def retrieve_with_window(query, top_n=100):
    sorted_pids, pid_to_idx = get_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, rrf_score in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        c = all_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce       = cross_encoder.predict([[query, s] for s in curr_sents])
        top_current_ce = float(max(curr_ce))
        window        = get_window(c['doc_name'], idx)
        win_sents     = sent_tokenize(window)
        win_ce        = cross_encoder.predict([[query, s] for s in win_sents])
        kept          = inner_decay(win_sents, win_ce)
        if kept:
            results.append({
                'doc_name':        c['doc_name'],
                'top_ce_score':    top_current_ce,
                'extracted_spans': [s for _, s in kept],
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

# ── WITHOUT window (anchor passage only) ─────────────────────
def retrieve_no_window(query, top_n=100):
    sorted_pids, pid_to_idx = get_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, rrf_score in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        c = all_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        kept    = inner_decay(curr_sents, curr_ce)  # current only
        if kept:
            results.append({
                'doc_name':        c['doc_name'],
                'top_ce_score':    top_ce,
                'extracted_spans': [s for _, s in kept],
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

# ── Evaluator ─────────────────────────────────────────────────
def evaluate_rqa(df, retrieve_fn):
    f1_scores, r03, r05 = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        query        = row['question']
        ground_truth = str(row['ground_truth_answer'])
        results      = retrieve_fn(query)
        best_f1      = 0.0
        for r in results[:3]:
            for span in r['extracted_spans']:
                f1 = token_f1(span, ground_truth)
                if f1 > best_f1:
                    best_f1 = f1
        f1_scores.append(best_f1)
        r03.append(1.0 if best_f1 >= 0.3 else 0.0)
        r05.append(1.0 if best_f1 >= 0.5 else 0.0)
    return {
        'Mean F1':    np.mean(f1_scores),
        'Recall@0.3': np.mean(r03),
        'Recall@0.5': np.mean(r05),
    }

# ── Run both ──────────────────────────────────────────────────
print("\nRunning: Full pipeline WITH window (n=1002)...")
r_window = evaluate_rqa(rqa_df, retrieve_with_window)

print("\nRunning: Full pipeline WITHOUT window (n=1002)...")
r_nowindow = evaluate_rqa(rqa_df, retrieve_no_window)

# ── Print comparison ──────────────────────────────────────────
print("\n=== WINDOW ABLATION — RegulationQA (n=1002) ===")
print(f"{'System':<35} {'F1':>7} {'R@0.3':>7} {'R@0.5':>7}")
print("-" * 58)
print(f"{'Full pipeline (with window)':<35} "
      f"{r_window['Mean F1']:>7.4f} "
      f"{r_window['Recall@0.3']:>7.4f} "
      f"{r_window['Recall@0.5']:>7.4f}")
print(f"{'Full pipeline (no window)':<35} "
      f"{r_nowindow['Mean F1']:>7.4f} "
      f"{r_nowindow['Recall@0.3']:>7.4f} "
      f"{r_nowindow['Recall@0.5']:>7.4f}")
delta_f1 = r_window['Mean F1'] - r_nowindow['Mean F1']
print(f"\nWindow contribution: ΔF1 = {delta_f1:+.4f}")

# ── Save ──────────────────────────────────────────────────────
import json
with open('/kaggle/working/window_ablation_rqa.json', 'w') as f:
    json.dump({
        'with_window':    r_window,
        'without_window': r_nowindow,
        'delta_f1':       delta_f1,
    }, f, indent=2)
print("\nSaved: window_ablation_rqa.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.0 MB/s eta 0:00:00:00:0100:01
Loading artifacts...
Chunks: 766, Questions: 1002


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (17 outputs left out)

100%|██████████| 1002/1002 [33:59<00:00,  2.04s/it]



Running: Full pipeline WITHOUT window (n=1002)...


100%|██████████| 1002/1002 [08:36<00:00,  1.94it/s]


=== WINDOW ABLATION — RegulationQA (n=1002) ===
System                                   F1   R@0.3   R@0.5
----------------------------------------------------------
Full pipeline (with window)          0.3598  0.4022  0.2745
Full pipeline (no window)            0.3551  0.3962  0.2685

Window contribution: ΔF1 = +0.0047

Saved: window_ablation_rqa.json


In [ ]:
import pickle, faiss, json, numpy as np, string, pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Load artifacts ────────────────────────────────────────────
with open('/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/chunks.pkl', 'rb') as f:
    all_chunks = pickle.load(f)
index = faiss.read_index(
    '/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/faiss_index.bin'
)
with open('/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/bm25.pkl', 'rb') as f:
    bm25 = pickle.load(f)
rqa_df = pd.read_csv(
    '/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv'
)
bge_model     = SentenceTransformer('BAAI/bge-base-en-v1.5')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f"Ready. Chunks: {len(all_chunks)}, Questions: {len(rqa_df)}")

# ── Helpers ───────────────────────────────────────────────────
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

def token_f1(prediction, ground_truth):
    pred = prediction.lower().split()
    gt   = ground_truth.lower().split()
    common = set(pred) & set(gt)
    if not common:
        return 0.0
    p = len(common) / len(pred)
    r = len(common) / len(gt)
    return 2 * p * r / (p + r)

def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window(doc_name, passage_idx):
    doc_chunks  = [(i, c) for i, c in enumerate(all_chunks)
                   if c['doc_name'] == doc_name]
    current_pos = next(
        (pos for pos, (i, c) in enumerate(doc_chunks)
         if i == passage_idx), None
    )
    if current_pos is None:
        return all_chunks[passage_idx]['text']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_chunks):
            window.append(doc_chunks[pos][1]['text'])
    return ' '.join(window)

def get_rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    _, bge_indices = index.search(qe, top_n)
    bge_indices  = bge_indices[0]
    bm25_scores  = bm25.get_scores(tokenize(query))
    bm25_indices = np.argsort(bm25_scores)[::-1][:top_n]
    rrf_scores   = {}
    for rank, idx in enumerate(bge_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(
        rrf_scores.items(), key=lambda x: x[1], reverse=True
    )[:20]
    pid_to_idx = {}
    for i, c in enumerate(all_chunks):
        pid = (c['doc_name'], c['chunk_idx'])
        if pid not in pid_to_idx:
            pid_to_idx[pid] = i
    return sorted_pids, pid_to_idx

def retrieve_with_window(query, top_n=100):
    sorted_pids, pid_to_idx = get_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        c          = all_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce    = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce     = float(max(curr_ce))
        window     = get_window(c['doc_name'], idx)
        win_sents  = sent_tokenize(window)
        win_ce     = cross_encoder.predict([[query, s] for s in win_sents])
        kept       = inner_decay(win_sents, win_ce)
        if kept:
            results.append({
                'doc_name':        c['doc_name'],
                'top_ce_score':    top_ce,
                'extracted_spans': [s for _, s in kept],
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_no_window(query, top_n=100):
    sorted_pids, pid_to_idx = get_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        c          = all_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        kept    = inner_decay(curr_sents, curr_ce)
        if kept:
            results.append({
                'doc_name':        c['doc_name'],
                'top_ce_score':    top_ce,
                'extracted_spans': [s for _, s in kept],
            })
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def evaluate_rqa_sample(df, retrieve_fn, n=200, seed=42):
    sample = df.sample(n, random_state=seed)
    f1_scores, r03, r05 = [], [], []
    for _, row in tqdm(sample.iterrows(), total=n):
        query        = row['question']
        ground_truth = str(row['ground_truth_answer'])
        results      = retrieve_fn(query)
        best_f1      = 0.0
        for r in results[:3]:
            for span in r['extracted_spans']:
                f1 = token_f1(span, ground_truth)
                if f1 > best_f1:
                    best_f1 = f1
        f1_scores.append(best_f1)
        r03.append(1.0 if best_f1 >= 0.3 else 0.0)
        r05.append(1.0 if best_f1 >= 0.5 else 0.0)
    return {
        'Mean F1':    np.mean(f1_scores),
        'Recall@0.3': np.mean(r03),
        'Recall@0.5': np.mean(r05),
    }

# ── 3 runs to check variance ──────────────────────────────────
print("\nChecking variance — 3 runs × n=200...")
print(f"\n{'Seed':<6} {'With Window F1':>16} {'No Window F1':>14} {'Delta':>8}")
print("-" * 48)

deltas = []
for seed in [42, 123, 999]:
    r_w  = evaluate_rqa_sample(rqa_df, retrieve_with_window, n=200, seed=seed)
    r_nw = evaluate_rqa_sample(rqa_df, retrieve_no_window,   n=200, seed=seed)
    delta = r_w['Mean F1'] - r_nw['Mean F1']
    deltas.append(delta)
    print(f"{seed:<6} {r_w['Mean F1']:>16.4f} {r_nw['Mean F1']:>14.4f} {delta:>+8.4f}")

print(f"\nMean delta: {np.mean(deltas):+.4f}")
print(f"All positive: {all(d > 0 for d in deltas)}")
print("\nConclusion:", 
      "window consistently helps" if all(d > 0 for d in deltas)
      else "difference is within noise")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready. Chunks: 766, Questions: 1002

Checking variance — 3 runs × n=200...

Seed     With Window F1   No Window F1    Delta
------------------------------------------------


100%|██████████| 200/200 [01:43<00:00,  1.93it/s]


42               0.3856         0.3867  -0.0011


100%|██████████| 200/200 [01:45<00:00,  1.90it/s]


123              0.3541         0.3506  +0.0035


 58%|█████▊    | 116/200 [04:04<02:55,  2.08s/it]